In [10]:
# Load in libraries and match data

import json
import pandas as pd

with open("Single Match Data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)

# Index to make sure we have correct order of events

df = df.sort_values("index").reset_index(drop=True)


In [11]:
df. head()

,id,index,period,timestamp,minute,second,possession,duration,type.id,type.name,...,shot.one_on_one,foul_committed.advantage,foul_won.advantage,clearance.aerial_won,pass.deflected,pass.no_touch,foul_committed.type.id,foul_committed.type.name,pass.straight,pass.goal_assist
0,9f6e2ecf-6685-45df-a62e-c2db3090f6c1,1,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0300039d-150d-41e4-b29a-76602ef002e6,2,1,00:00:00.000,0,0,1,0.000000,35,Starting XI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,491e8901-7630-4cc8-b57b-937dddff2eaa,3,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,757b85ad-ddfe-44d5-b893-c23a9fb709d8,4,1,00:00:00.000,0,0,1,0.000000,18,Half Start,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,5,1,00:00:00.575,0,0,2,2.015669,30,Pass,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df["type.name"].value_counts()

type.name
Pass              1163
Ball Receipt*     1058
Carry              890
Pressure           212
Ball Recovery       89
Duel                53
Clearance           37
Goal Keeper         34
Block               32
Shot                28
Interception        24
Dribble             24
Foul Committed      23
Dispossessed        21
Foul Won            21
Miscontrol          17
Dribbled Past       14
Substitution         6
Half Start           4
Half End             4
Tactical Shift       4
Starting XI          2
Bad Behaviour        1
Error                1
Name: count, dtype: int64

In [13]:
# Don't want location like this [61.0, 40.1]. Want a separate x and y column

df["x"] = df["location"].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None
)

df["y"] = df["location"].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None
)

In [14]:
df["type.name"].value_counts()


type.name
Pass              1163
Ball Receipt*     1058
Carry              890
Pressure           212
Ball Recovery       89
Duel                53
Clearance           37
Goal Keeper         34
Block               32
Shot                28
Interception        24
Dribble             24
Foul Committed      23
Dispossessed        21
Foul Won            21
Miscontrol          17
Dribbled Past       14
Substitution         6
Half Start           4
Half End             4
Tactical Shift       4
Starting XI          2
Bad Behaviour        1
Error                1
Name: count, dtype: int64

In [15]:
df["possession"].nunique()


143

In [16]:
# See what shots look like for EDA

shots = df[df["type.name"] == "Shot"].copy()

shots[[
    "team.name",
    "player.name",
    "possession",
    "shot.statsbomb_xg",
    "shot.outcome.name"
]]

,team.name,player.name,possession,shot.statsbomb_xg,shot.outcome.name
136,Barcelona,Lionel Andrés Messi Cuccittini,6,0.076992,Off T
261,Barcelona,Jordi Alba Ramos,12,0.051668,Off T
714,Barcelona,Lionel Andrés Messi Cuccittini,23,0.016932,Saved
742,Deportivo Alavés,Rubén Sobrino Pozuelo,30,0.122604,Off T
801,Barcelona,Luis Alberto Suárez Díaz,33,0.041751,Off T
1340,Barcelona,Ousmane Dembélé,50,0.076063,Wayward
1546,Barcelona,Ivan Rakitić,58,0.112513,Off T
1587,Barcelona,Lionel Andrés Messi Cuccittini,61,0.049074,Post
1591,Barcelona,Gerard Piqué Bernabéu,61,0.115620,Off T
1619,Barcelona,Ousmane Dembélé,63,0.123355,Saved


In [ ]:
# These are the columns we want

cols = [
    "index",
    "possession",
    "possession_team.name",
    "team.name",
    "minute",
    "second",
    "player.name",
    "type.name",
    "x",
    "y",
    "shot.statsbomb_xg",
    "shot.outcome.name"
]


# Take out usesless values

df = df[
    ~df["type.name"].isin([
        "Half Start",
        "Starting XI",
        "Half End",
        "Substitution",
        "Tactical Shift"
    ])
].copy()

# Only want actions done by the team in possession for now

df = df[
    df["team.name"] == df["possession_team.name"]
].copy()

# Half Start and Starting XI counted as possession 1.
# Make the first true possession = 1

df["possession"] = df["possession"] - 1

# Number events within each possession performed by team in possession

df["possession_event_num"] = (
    df.groupby("possession").cumcount() + 1
)

# Get each action's OWN end location - where the ball actually ended up
# because of this action - rather than inferring it from wherever the
# next logged row happens to start.
# e.g. if Messi passes from (20, 20) to (50, 50), we want (50, 50)
# attributed to Messi's pass directly, not borrowed from the next event.

end_location_cols = [
    "pass.end_location",
    "carry.end_location",
    "shot.end_location",
    "goalkeeper.end_location"
]

def get_end_xy(row):

    for col in end_location_cols:

        # .get() prevents an error if a particular StatsBomb
        # dataset does not contain this column
        val = row.get(col)

        if isinstance(val, list):

            # Some shot end locations contain a third value
            # representing height. We only want x and y.
            return val[0], val[1]

    # Point event with no separately recorded end location
    return row["x"], row["y"]

end_xy = df.apply(
    get_end_xy,
    axis=1,
    result_type="expand"
)

df["end_x"] = end_xy[0]
df["end_y"] = end_xy[1]

In [ ]:
# Shot -> reward = StatsBomb xG
# Everything else -> reward = 0

# We use shot quality (xG), NOT whether the shot actually went in.

# A 0.70 xG shot is still a valuable opportunity
# even if the player misses.

df["reward"] = (
    df["shot.statsbomb_xg"]
    .fillna(0)
)

In [ ]:
# future_xg = total xG generated from the current event
# through the remainder of the possession.

# future_xg is NOT the player's value.

# It is the observed outcome we will use to teach a model
# what dangerous states tend to look like.

# Eventually: V(s) = expected future attacking value from state s

df["future_xg"] = (
    df.groupby(
        ["possession", "possession_team.name"]
    )["reward"]
    .transform(
        lambda s: s.iloc[::-1].cumsum().iloc[::-1]
    )
)